# Feedback Control: From Bang-Bang to PID

Welcome to Week 1 of *Autonomous Systems and Mobile Robots*. Last week you implemented the full **Perceive → Reason → Act** loop for a discrete grid robot. This week you zoom in on the **Act** step: given a sensed error from the Perceive step, how does the robot compute and apply an actuator command to reduce it?

The running example is taken directly from the lecture: an autonomous car that must stay in a lane at fixed forward speed. You will implement four controllers of increasing sophistication, observe their failure modes, and compare them side by side.

### What you will do

| Task | Controller | Key concept |
|------|------------|-------------|
| 1 | Bang-bang | Binary output — cannot scale with error magnitude |
| 2 | P | Proportional scaling; gain trade-off |
| 3 | PD | Derivative damping; parameter exploration |
| 4 | — | Systematic vs. random error |
| 5 | PID | Integral term; integrator windup and anti-windup |
| 6 | All | Comparison and recap |

**Prerequisites** — Week 0 exercise (PRA loop, Python / NumPy / Matplotlib).

---

## Task 0 — The Control Problem

### Feedback control in the PRA loop

This notebook focuses on the **Act** step of the Perceive → Reason → Act loop applied to a lane-following car:

- **Perceive** — a sensor measures the lateral distance between the car and the target lane. This is the *cross-track error* $e(t) = y_\mathrm{ref} - y(t)$.
- **Reason** — higher-level planning (path selection, goal setting) is assumed solved. The goal is fixed: stay on the lane.
- **Act** — a controller receives $e(t)$ and computes a steering command $\omega(t)$, which the actuator applies.

Tasks 1–5 each ask the same question from the Act step: *given $e(t)$, what is the best command $\omega(t)$?*

### Simulation model

| Symbol | Meaning |
|--------|---------|
| $v$ | Forward speed — **fixed**, not controlled |
| $\omega$ | Heading rate — the **control output** (rad/s) |
| $\omega_{\max}$ | Actuator limit: $\omega \in [-\omega_{\max},\, \omega_{\max}]$ |
| $e(t)$ | Cross-track error: signed lateral distance to target line (m) |
| $\theta$ | Car heading angle (rad) |

### Unicycle kinematics

The car is modelled as a **unicycle** — a standard abstraction for wheeled robots that can only move in the direction they face (no sideways slip). The state is the pose $(x, y, \theta)$: position plus heading angle.

The velocity vector is always aligned with the heading, giving the **continuous-time equations of motion**:

$$\dot{x}(t) = v\cos\theta(t), \qquad \dot{y}(t) = v\sin\theta(t), \qquad \dot{\theta}(t) = \omega(t)$$

The simulator uses **Euler integration** with timestep $\Delta t$:

$$\theta_{k+1} = \theta_k + \omega_k \cdot \Delta t, \qquad x_{k+1} = x_k + v\cos(\theta_k)\cdot\Delta t, \qquad y_{k+1} = y_k + v\sin(\theta_k)\cdot\Delta t$$

$$e_k = y_\mathrm{ref} - y_k$$

The car starts at $y_0 = 1\,\mathrm{m}$ above the target line, heading straight ($\theta_0 = 0$). Your goal: implement a controller that drives $e(t) \to 0$.

The simulation and plotting infrastructure is pre-written in `utils.py`. Run the cell below to load it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils import (
    V, DT, STEPS, Y0, Y_REF, W_MAX, DIST, NOISE_STD,
    simulate,
    plot_trajectory,
    plot_signals,
)

print("Simulation ready.")
print(f"  v={V} m/s  ·  dt={DT} s  ·  steps={STEPS}  ·  y0={Y0} m  ·  ω_max={W_MAX} rad/s")
print(f"  systematic error magnitude={DIST} m/s  ·  random error σ={NOISE_STD}")

All controllers share the same signature — the simulator calls them at every timestep:

```python
def my_controller(e, e_prev, integral, dt) -> float:
    ...
```

| Argument | Meaning |
|----------|---------|
| `e` | Current cross-track error (m) — may include random error |
| `e_prev` | Error at the previous timestep (m) |
| `integral` | Running sum $\sum_k e_k \cdot \Delta t$ accumulated by the simulator |
| `dt` | Timestep $\Delta t$ (s) |

You will not modify the simulator. Focus entirely on implementing the controller function.

---

## Task 1 — Bang-Bang Control

The simplest possible Act strategy: apply maximum steering effort in the direction of the error, regardless of how large the error is.

$$\omega = \begin{cases} \omega_{\max} & \text{if } e(t) > 0 \\ -\omega_{\max} & \text{if } e(t) < 0 \\ 0 & \text{if } e(t) = 0 \end{cases}$$

**Core problem:** the output is **binary** — it cannot scale with error magnitude. Whether the car is 2 m or 2 mm from the target line, it receives the same maximum steering command. Every approach therefore overshoots, triggering an immediate full reversal. This persistent oscillation is called *chattering*.

<div class='alert alert-warning'>
<b>Exercise 1 — Bang-bang controller</b>

Implement <code>bang_bang</code> using the equation above. Use the module-level constant <code>W_MAX</code> as $\omega_{\max}$.
</div>

In [ ]:
def bang_bang(e, e_prev, integral, dt):
    # ---- TODO: return +W_MAX, -W_MAX, or 0 depending on the sign of e ----
    omega = ...  # replace
    # ---- END TODO ---------------------------------------------------------
    return omega

In [ ]:
assert bang_bang( 1.0, 0.0, 0.0, DT) ==  W_MAX, "Positive error → +W_MAX"
assert bang_bang(-1.0, 0.0, 0.0, DT) == -W_MAX, "Negative error → -W_MAX"
assert bang_bang( 0.0, 0.0, 0.0, DT) ==  0.0,   "Zero error → 0"
print("bang_bang checks passed.")

In [ ]:
t, x, y, e, omega = simulate(bang_bang)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x, y, "Bang-bang — trajectory", ax=axes[0])
plot_signals(t, e, omega, "Bang-bang — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

> **Reflection 1** — Describe the trajectory. Does the car ever settle on the target line? Look at the $\omega(t)$ plot: what pattern do you see, and why does it arise from a binary controller?

---

## Task 2 — Proportional (P) Control

Task 1 showed that a binary command cannot reduce error gradually. Proportional control scales the output directly with the error magnitude:

$$\omega(t) = K_p \, e(t)$$

The gain $K_p$ controls how aggressively the car steers:

- $K_p$ too small → slow convergence.
- $K_p$ too large → overshoot and oscillation.

The controller reacts to *how far* the car is from the line, but not to *how fast* it is approaching. This leaves overshoot as an open problem at high gains.

<div class='alert alert-warning'>
<b>Exercise 2a — P controller</b>

Implement <code>p_controller</code>. Use <code>Kp = 0.4</code> as the starting value. Observe the <em>shape</em> of the response — how does it differ from bang-bang?
</div>

In [ ]:
Kp = 0.4

def p_controller(e, e_prev, integral, dt):
    # ---- TODO: return Kp * e ----
    omega = ...  # replace
    # ---- END TODO ---------------
    return omega

In [ ]:
assert p_controller(2.0, 0.0, 0.0, DT) == 2.0 * p_controller(1.0, 0.0, 0.0, DT), \
    "Output must scale linearly with error"
assert p_controller(0.0, 0.0, 0.0, DT) == 0.0, "Zero error must give zero output"
print("p_controller checks passed.")

In [ ]:
t, x, y, e, omega = simulate(p_controller)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x, y, f"P control (Kp={Kp}) — trajectory", ax=axes[0])
plot_signals(t, e, omega, f"P control (Kp={Kp}) — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

> **Reflection 2a** — Does the car settle on the target line, or does it oscillate? How does the shape of the $\omega(t)$ signal differ from bang-bang?

<div class='alert alert-warning'>
<b>Exercise 2b — Gain exploration</b>

Set <code>kp_values</code> to three values: one very small, one well-tuned, and one large that causes saturation. The cell overlays all three cross-track error curves.
</div>

In [ ]:
kp_values = [..., ..., ...]  # ---- TODO: fill in three Kp values ----

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].axhline(Y_REF, color="green", lw=1.5, ls="--", label="Target")
axes[1].axhline(0, color="green", lw=1.0, ls="--")

colors = plt.cm.viridis([0.15, 0.5, 0.85])
for kp, col in zip(kp_values, colors):
    Kp = kp
    t, x, y, e, omega = simulate(p_controller)
    axes[0].plot(x, y, color=col, lw=1.5, label=f"Kp={kp}")
    axes[1].plot(t, e, color=col, lw=1.5, label=f"Kp={kp}")

axes[0].set(xlabel="x (m)", ylabel="y (m)", ylim=(-3, 3), title="Trajectory")
axes[1].set(xlabel="t (s)", ylabel="e(t) (m)", title="Cross-track error")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

> **Reflection 2b** — Which $K_p$ produced oscillation, and why? What information does P control have access to, and what does it not know?

---

## Task 3 — Proportional-Derivative (PD) Control

The derivative term adds a correction proportional to the *rate of change* of the error:

$$\omega(t) = K_p \, e(t) + K_d \, \dot{e}(t)$$

If the error is already decreasing (the car is turning toward the line), the D term reduces the command — it acts as a *damper* that anticipates overshoot. In discrete time:

$$\dot{e}(t) \approx \frac{e(t) - e(t - \Delta t)}{\Delta t}$$

Two parameters must be tuned: $K_p$ (reaction to current error magnitude) and $K_d$ (reaction to error trend).

<div class='alert alert-warning'>
<b>Exercise 3a — PD controller</b>

Implement <code>pd_controller</code>. Use <code>Kp = 0.4</code> and <code>Kd = 1.0</code> as starting values. Use the <code>e_prev</code> argument to compute the finite-difference derivative. Compare the result with Task 2a.
</div>

In [1]:
Kp = 0.4
Kd = 1.

def pd_controller(e, e_prev, integral, dt):
    # ---- TODO: compute derivative and return Kp*e + Kd*derivative ----
    derivative = ...  # (e - e_prev) / dt
    omega      = ...  # Kp * e + Kd * derivative
    # ---- END TODO -------------------------------------------------------
    return omega

In [ ]:
# When error is constant (e == e_prev), the derivative is zero → PD equals pure P
omega_pd = pd_controller(1.0, 1.0, 0.0, DT)
assert abs(omega_pd - Kp * 1.0) < 1e-9, "With constant error, PD must equal pure P output"
print("pd_controller checks passed.")

In [ ]:
t, x, y, e, omega = simulate(pd_controller)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x, y, f"PD (Kp={Kp}, Kd={Kd}) — trajectory", ax=axes[0])
plot_signals(t, e, omega, f"PD (Kp={Kp}, Kd={Kd}) — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

> **Reflection 3a** — Compare the trajectory and $e(t)$ with Task 2a (same $K_p$). How does adding $K_d$ change the behaviour?

<div class='alert alert-warning'>
<b>Exercise 3b — Parameter exploration</b>

Keep <code>Kp = 0.4</code> fixed. Set <code>kd_values</code> to three values: one very small , one well-tuned , and one very large. Observe how each regime behaves differently.
</div>

In [ ]:
Kp = 0.4
kd_values = [..., ..., ...]  # ---- TODO: fill in three Kd values ----

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].axhline(Y_REF, color="green", lw=1.5, ls="--", label="Target")
axes[1].axhline(0, color="green", lw=1.0, ls="--")

colors = plt.cm.plasma([0.15, 0.5, 0.85])
for kd, col in zip(kd_values, colors):
    Kd = kd
    t, x, y, e, omega = simulate(pd_controller)
    axes[0].plot(x, y, color=col, lw=1.5, label=f"Kd={kd}")
    axes[1].plot(t, e, color=col, lw=1.5, label=f"Kd={kd}")

axes[0].set(xlabel="x (m)", ylabel="y (m)", ylim=(-3, 3), title="Trajectory")
axes[1].set(xlabel="t (s)", ylabel="e(t) (m)", title="Cross-track error")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Task 4 — Systematic and Random Error

So far the simulation has run under ideal conditions. In practice, two qualitatively different error sources affect what the controller receives from the Perceive step:

**Systematic error** is a constant, repeatable bias applied at every timestep — in this simulation, a steady angular disturbance that persistently turns the car away from the lane (like a crosswind yaw torque). It is always present in the same direction and magnitude, so its effect accumulates. This corresponds to *bias*: the controller receives a consistently wrong picture of the world.

**Random error** is a zero-mean, unpredictable fluctuation added to the sensor reading at each timestep. Individual samples are unreliable, but averaged over time they cancel. This corresponds to *variance*: the controller is right on average but receives a scattered signal.

The distinction matters because they call for fundamentally different remedies — as you will see in Task 5.

<div class='alert alert-warning'>
<b>Exercise 4a — PD under systematic error</b>

Run your <code>pd_controller</code> with <code>systematic_error=True</code>. Read the final cross-track error from the printed value. Does the car reach $e = 0$?
</div>

In [ ]:
t, x, y, e, omega = simulate(pd_controller, systematic_error=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x, y, "PD + systematic error — trajectory", ax=axes[0])
plot_signals(t, e, omega, "PD + systematic error — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

print(f"Final cross-track error: {e[-1]:.4f} m")

> **Reflection 4a** — Does the car reach $e = 0$? Why does PD, which converged cleanly in Task 3a, leave a persistent offset here? What property of PD prevents it from cancelling a constant angular bias?

<div class='alert alert-warning'>
<b>Exercise 4b — PD under random error</b>

Run <code>pd_controller</code> with <code>random_error=True</code>. Focus on the $\omega(t)$ signal.

Which term in the PD law amplifies noise most strongly? The D term computes $(e - e_\mathrm{prev}) / \Delta t$. If $e$ carries noise with standard deviation $\sigma$, what is the approximate standard deviation of the finite-difference output?
</div>

In [ ]:
t, x, y, e, omega = simulate(pd_controller, random_error=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x, y, "PD + random error — trajectory", ax=axes[0])
plot_signals(t, e, omega, "PD + random error — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

> **Reflection 4b** — What happened to the trajectory and the $\omega(t)$ signal under random error? Does the car converge as in Task 3a? What could be possible solutions?

---

## Task 5 — Proportional-Integral-Derivative (PID) Control

The integral term gives the controller a *memory* of accumulated past error. A constant systematic error causes the integral to grow until its contribution is large enough to cancel the bias — at which point the car reaches the target and the integral stops growing.

$$\omega(t) = K_p \, e(t) \;+\; K_i \int_0^{t} e(\tau)\, d\tau \;+\; K_d \, \dot{e}(t)$$

The simulator accumulates $\sum_k e_k \cdot \Delta t$ and passes it as the `integral` argument.

**Integrator windup** is a failure mode that occurs when the car starts far from the target and $\omega$ is saturated. During saturation the actuator cannot respond to increasing $\omega$, yet the integral keeps growing. When the car eventually crosses the target line, the large stored value drives $\omega$ in the wrong direction — causing severe overshoot.

<div class='alert alert-warning'>
<b>Exercise 5a — PID under systematic error</b>

Implement <code>pid_controller</code>. Run with <code>systematic_error=True</code> and verify that the steady-state offset from Task 4a disappears. Start with a small $K_i$ (e.g. 0.01).
</div>

In [ ]:
Kp = 1.0
Ki = ...  # ---- TODO: choose a value, e.g. 0.2 ----
Kd = 1.5

def pid_controller(e, e_prev, integral, dt):
    # ---- TODO: implement PID — use the integral argument directly ----
    derivative = ...  # (e - e_prev) / dt
    omega      = ...  # Kp * e + Ki * integral + Kd * derivative
    # ---- END TODO ---------------------------------------------------
    return omega

In [ ]:
t, x, y, e, omega = simulate(pid_controller, systematic_error=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x, y, f"PID (Kp={Kp}, Ki={Ki}, Kd={Kd}) + systematic error", ax=axes[0])
plot_signals(t, e, omega, "PID + systematic error — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

print(f"Final cross-track error: {e[-1]:.4f} m")

> **Reflection 5a** — The integral accumulates cross-track error over time. In one sentence: why does this accumulated value eventually equal exactly what is needed to cancel a constant systematic error at steady state?

<div class='alert alert-warning'>
<b>Exercise 5b — Integrator windup (guided observation)</b>

We now start the car further from the target ($y_0 = 4\,\mathrm{m}$) so that $\omega$ is saturated for longer. Run the cell below and observe what happens when the car crosses the target line.

Please set the parameter values of yout pid-controller as follows:

Kp = 0.4, Ki = 0.1, Kd = 1.

</div>

In [ ]:
# Pre-written — run and observe
t_w, x_w, y_w, e_w, omega_w = simulate(pid_controller, systematic_error=True, y0=4.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_trajectory(x_w, y_w, "PID (y0=3 m) — observe the overshoot", ax=axes[0])
plot_signals(t_w, e_w, omega_w, "PID — signals", ax_e=axes[1], ax_w=axes[2])
plt.tight_layout()
plt.show()

Notice the large overshoot after the car crosses the target line. While the car was far above the target, $\omega$ was clamped at $-\omega_{\max}$, yet the integral kept accumulating positive error. When the car finally crossed, the stored integral drove $\omega$ negative even though the car had already passed the target — causing it to overshoot significantly.

**Fix — anti-windup by clamping:** prevent the integral from growing beyond $\pm I_{\max}$ before using it in the control law. This limits how much the integral can accumulate while the actuator is saturated.

<div class='alert alert-warning'>
<b>Exercise 5b (continued) — Implement anti-windup</b>

Complete <code>pid_antiwindup</code> below. The only change from <code>pid_controller</code>: clamp the integral with <code>np.clip(integral, -I_MAX, I_MAX)</code> before computing the output.
</div>

In [ ]:
I_MAX = 2.0  # anti-windup clamp threshold

def pid_antiwindup(e, e_prev, integral, dt):
    # ---- TODO: clamp the integral, then compute omega ----
    integral_clamped = ...  
    derivative       = ...  
    omega            = ...  
    # ---- END TODO ----------------------------------------
    return omega

In [ ]:
t_aw, x_aw, y_aw, e_aw, omega_aw = simulate(pid_antiwindup, systematic_error=True, y0=4.0)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
plot_trajectory(x_w,  y_w,  "PID basic (y0=4 m) — windup",      ax=axes[0, 0])
plot_signals(t_w,  e_w,  omega_w,  "PID basic — signals",        ax_e=axes[0, 1], ax_w=axes[0, 2])
plot_trajectory(x_aw, y_aw, "PID anti-windup (y0=4 m)",          ax=axes[1, 0])
plot_signals(t_aw, e_aw, omega_aw, "PID anti-windup — signals",  ax_e=axes[1, 1], ax_w=axes[1, 2])
plt.tight_layout()
plt.show()

> **Reflection 5b** — At what moment in the trajectory does windup cause the problem? How does clamping the integral prevent this?

---

## Task 6 — Controller Comparison and Recap

The cell below runs all four controllers under identical clean conditions (no systematic or random error, $y_0 = 2\,\mathrm{m}$) and overlays the results.

In [ ]:
results = {
    "Bang-bang": simulate(bang_bang),
    "P":         simulate(p_controller),
    "PD":        simulate(pd_controller),
    "PID":       simulate(pid_antiwindup),
}

colors = {"Bang-bang": "tomato", "P": "goldenrod", "PD": "steelblue", "PID": "mediumseagreen"}

fig, (ax_traj, ax_e) = plt.subplots(1, 2, figsize=(14, 4))
ax_traj.axhline(Y_REF, color="green", lw=1.5, ls="--", label="Target")
ax_e.axhline(0, color="green", lw=1.0, ls="--")

for label, (t, x, y, e, omega) in results.items():
    ax_traj.plot(x, y, label=label, color=colors[label], lw=1.5)
    ax_e.plot(t, e,    label=label, color=colors[label], lw=1.5)

ax_traj.set(xlabel="x (m)", ylabel="y (m)", ylim=(-3, 3),
            title="Trajectory comparison (clean conditions)")
ax_e.set(xlabel="t (s)", ylabel="e(t) (m)",
         title="Cross-track error comparison (clean conditions)")
ax_traj.legend(fontsize=9)
ax_e.legend(fontsize=9)
plt.tight_layout()
plt.show()

<div class='alert alert-warning'>
<b>Exercise 6 — Summary table</b>

Fill in the table based on your observations from Tasks 1–5. Mark each cell ✓ (yes), ✗ (no), or ~ (partially / depends on tuning).
</div>

| Property | Bang-bang | P | PD | PID |
|----------|-----------|---|----|-----|
| Reaches target line (clean) | | | | |
| Settles without oscillation (clean) | | | | |
| Zero steady-state error (systematic error) | | | | |
| Smooth actuation under random error | | | | |
| Requires gain tuning | | | | |
| **Known failure mode** | chattering | overshoot at high $K_p$ | noise amplification in $\omega$ | integrator windup |

> The last row is pre-filled. Use it to cross-check your answers.